In [ ]:
!pip install -q ultralytics 

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb-key")

# Training 

In [ ]:
from ultralytics import YOLO
from ultralytics import settings
import torch
import wandb 

In [ ]:
def train_yolo(model_path, data_yaml_path):

    wandb.login(key=wandb_key)
    
    run = wandb.init(
        project="fire-detection",
        name="yolo11-fire-detection",
    )
    
    model = YOLO(model_path) 
    settings.update({"wandb": True})

    
    results = model.train(
        data=data_yaml_path,  
        epochs=20,            
        imgsz=640,           
        batch=16,            
        workers=4,            
        device=0,  
        project='fire-detection-yolo',  
        name='yolo11s',       
        save=True,            
        verbose=True,       
        seed=42, 
        freeze = 10,



        optimizer='AdamW', 
        lr0 = 0.001, 
        lrf=0.01, 
        momentum=0.937, 
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        pretrained=True
    )

   
    
    val_results = model.val()
    metrics = val_results.box
    
    wandb.log({
        "val/mAP50": metrics.map50,      
        "val/mAP50-95": metrics.map,     
        "val/precision": metrics.mp,    
        "val/recall": metrics.mr,        
    })
    print(f"Validation results: {val_results}")

    
    wandb.finish()
    
    return model, results

In [ ]:
import yaml 
import os 


def load_and_modify_yaml(yaml_path, modifications=None):
    
    with open(yaml_path, 'r') as file:
        data = yaml.safe_load(file)
    
    print("Original YAML content:")
    print(yaml.dump(data, default_flow_style=False))
    
    # 2. Modify data nếu có
    if modifications:
        for key, value in modifications.items():
            data[key] = value
            
    
    return data

def save_yaml(data, output_path):
  
    with open(output_path, 'w') as file:
        yaml.dump(data, file, default_flow_style=False)
    print(f"Saved modified YAML to: {output_path}")

In [ ]:
modify = {
    'path' : '/kaggle/input/smoke-fire-detection-yolo'
}

data = load_and_modify_yaml('/kaggle/input/smoke-fire-detection-yolo/data.yaml', modify)
save_yaml(data, '/kaggle/working/data_config.yaml' )


In [ ]:
 train_yolo('/kaggle/input/yolo11/pytorch/default/1/yolo11s.pt', '/kaggle/working/data_config.yaml')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

def visualize_yolo_predictions(model, base_path, ids, conf_threshold=0.25, figsize=(12, 12), ground_truth: bool = None):
    
    image_path = base_path + '/images/' + ids + '.jpg'
    results = model.predict(image_path, conf=conf_threshold)
    result = results[0]

    
    
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    
    colors = ['red', 'blue', 'green', 'yellow']
    
    
    class_names = {
        0: "smoke",
        1: "fire", 
    }
    
    boxes = result.boxes.xyxy.cpu().numpy() 
    confs = result.boxes.conf.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    if ground_truth: 
        ground_truth_path = base_path + '/labels/' + ids + '.txt'
        
        if os.path.exists(ground_truth_path):
            with open(ground_truth_path, 'r') as f:
                annotations = f.readlines()
            
            
            for line in annotations:
                parts = line.strip().split()
                label = int(parts[0]) 
                x1, y1, x2, y2 = map(float, parts[1:])
                
                color = colors[(label + 1) % len(colors)]
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor='none')  
                ax.add_patch(rect)

                class_name = class_names.get(label, f"class_{label}")
                label = f"{class_name}: Ground Truth"
                ax.text(
                    x1, y1-10, label, fontsize=12, color='white', 
                    bbox=dict(facecolor=color, alpha=0.8, edgecolor='none', pad=3)
                )

    
    
    for box, conf, class_id in zip(boxes, confs, class_ids):
        x1, y1, x2, y2 = box
        

        color = colors[class_id % len(colors)]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor='none')        
        ax.add_patch(rect)
        
        class_name = class_names.get(class_id, f"class_{class_id}")
        label = f"{class_name}: {conf:.2f}"
        ax.text(
            x1, y1-10, label, fontsize=12, color='white', 
            bbox=dict(facecolor=color, alpha=0.8, edgecolor='none', pad=3)
        )
    
    ax.set_title(f"YOLO Predictions - {len(boxes)} objects detected", fontsize=16)
    ax.axis('off')
  
    plt.tight_layout()
    plt.show()
    
    return fig, ax



In [ ]:
model = YOLO('/kaggle/working/fire-detection-yolo/yolo11s3/weights/best.pt')

In [ ]:
base_path = '/kaggle/input/smoke-fire-detection-yolo/data/test'
ids = 'AoF06723'

visualize_yolo_predictions(model, base_path, ids, ground_truth = True)

In [ ]:
ids = 'AoF06726'

visualize_yolo_predictions(model, base_path, ids, ground_truth = True)

In [ ]:
ids = 'AoF06752'

visualize_yolo_predictions(model, base_path, ids, ground_truth = True)

In [ ]:
ids = 'AoF06748'

visualize_yolo_predictions(model, base_path, ids, ground_truth = True)